# 03-02 分词与 Embedding

**面试必考！** Tokenization 和 Embedding 是 LLM 的「感知器官」，RAG 和 Agent 开发都要深度依赖 Embedding。

**本节目标**：
- 理解 BPE / WordPiece 分词原理
- 掌握 Embedding 的本质（语义压缩到向量空间）
- 实战：文本相似度、语义搜索
- 了解 Embedding 模型的选型（OpenAI / 本地 sentence-transformers）

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = ['DejaVu Sans']

np.random.seed(42)

## 1. 分词（Tokenization）原理

LLM 不直接处理字符，而是处理 **token**（子词单元）。

### BPE (Byte Pair Encoding)
GPT 系列使用 BPE：从字符出发，反复合并频率最高的相邻字符对。

In [ ]:
# 手动演示 BPE 核心步骤
from collections import Counter, defaultdict

def get_vocab(corpus: list[str]) -> dict:
    """将语料转为 BPE 初始词汇表（每个词拆成字符，末尾加</w>）"""
    vocab = Counter()
    for text in corpus:
        for word in text.split():
            chars = list(word) + ['</w>']
            vocab[' '.join(chars)] += 1
    return dict(vocab)

def get_pairs(vocab: dict) -> Counter:
    """统计所有相邻字符对的频率"""
    pairs = Counter()
    for word, count in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i+1])] += count
    return pairs

def merge_vocab(pair: tuple, vocab: dict) -> dict:
    """合并最频繁的字符对"""
    new_vocab = {}
    bigram = ' '.join(pair)
    replacement = ''.join(pair)
    for word, count in vocab.items():
        new_word = word.replace(bigram, replacement)
        new_vocab[new_word] = count
    return new_vocab

# 模拟广告文案语料
corpus = [
    "click rate high low",
    "click through rate",
    "conversion rate low",
    "high conversion low cost",
]

vocab = get_vocab(corpus)
print("初始词表（字符级）:")
for word, cnt in list(vocab.items())[:4]:
    print(f"  '{word}': {cnt}")

print("\n--- BPE 合并过程 ---")
for i in range(6):
    pairs = get_pairs(vocab)
    if not pairs:
        break
    best = max(pairs, key=pairs.get)
    vocab = merge_vocab(best, vocab)
    print(f"第 {i+1} 次合并: {best[0]} + {best[1]} → {''.join(best)} (频率={pairs[best]})")

In [ ]:
# 用 tiktoken 演示 GPT-4 实际分词（需要 pip install tiktoken）
try:
    import tiktoken
    enc = tiktoken.encoding_for_model("gpt-4")
    
    samples = [
        "B站广告点击率分析",
        "click-through rate optimization",
        "Hello World!",
        "128000 tokens context window",
    ]
    
    print(f"{'文本':<30} {'token数':>8} {'tokens'}")
    print("-" * 70)
    for text in samples:
        tokens = enc.encode(text)
        decoded = [enc.decode([t]) for t in tokens]
        print(f"{text:<30} {len(tokens):>8}   {decoded}")
        
    print("""
关键发现：
  - 中文每个字 ≈ 1-2 tokens，英文每个词 ≈ 1 token
  - 数字通常每位 1 token（128000 → 4 tokens）
  - 特殊字符可能 1 token
  - 同样语义的文本，中文 token 数 ≈ 英文的 1.5-2x（API 调用中文更贵！）
    """)
except ImportError:
    print("tiktoken 未安装，跳过实际 GPT-4 分词演示")
    print("安装方式: pip install tiktoken")
    print("""
BPE 分词要点（面试用）：
  - GPT 系列: cl100k_base（GPT-4）, p50k_base（GPT-3）
  - BERT/中文模型: WordPiece（从完整词出发，拆分低频词）
  - LLaMA/Qwen: SentencePiece 的 BPE 变体
  - 中文: 通常字符级别（每字 1-2 tokens）
  - 空格通常属于下一个 token（GPT-2 用 Ġ 标记）
    """)

## 2. Word Embedding 本质

Embedding = **把离散 token 映射到连续向量空间**，使得语义相近的词距离近。

$$\text{embedding}(\text{"点击"}) \approx \text{embedding}(\text{"click"})$$
$$\text{embedding}(\text{"王"}) - \text{embedding}(\text{"男"}) \approx \text{embedding}(\text{"女王"}) - \text{embedding}(\text{"女"})$$

In [ ]:
# 用随机向量模拟 Embedding 空间（实际是模型学习的）
# 人工构造一些有语义关系的向量
np.random.seed(0)
d = 8  # 简化维度（实际 768-4096 维）

# 广告相关词的模拟 Embedding
words = ["点击", "CTR", "转化", "CVR", "曝光", "展示", "广告", "投放", "成本", "消耗"]
# 手动让语义相近的词有相似向量
base_click = np.random.randn(d)
base_convert = np.random.randn(d)
base_show = np.random.randn(d)
base_cost = np.random.randn(d)

embeddings = {
    "点击":  base_click + np.random.randn(d) * 0.1,
    "CTR":   base_click + np.random.randn(d) * 0.15,  # CTR 和 点击 相近
    "转化":  base_convert + np.random.randn(d) * 0.1,
    "CVR":   base_convert + np.random.randn(d) * 0.12,
    "曝光":  base_show + np.random.randn(d) * 0.1,
    "展示":  base_show + np.random.randn(d) * 0.08,
    "广告":  np.random.randn(d),
    "投放":  np.random.randn(d),
    "成本":  base_cost + np.random.randn(d) * 0.1,
    "消耗":  base_cost + np.random.randn(d) * 0.12,
}

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# 计算所有词对的相似度
print("语义相似度（余弦相似度）:")
pairs_to_check = [
    ("点击", "CTR"), ("转化", "CVR"), ("曝光", "展示"),
    ("成本", "消耗"), ("点击", "转化"), ("广告", "投放")
]
for w1, w2 in pairs_to_check:
    sim = cosine_similarity(embeddings[w1], embeddings[w2])
    print(f"  sim({w1!r:4}, {w2!r:4}) = {sim:.3f}")

## 3. 句子 Embedding 与语义搜索

**句子 Embedding** = 整个句子压缩成一个向量，用于：
- RAG 的文档检索
- Agent 的记忆检索
- 语义去重
- 分类

In [ ]:
# 用 sentence-transformers 做语义搜索（优先本地，无 GPU 也能跑）
try:
    from sentence_transformers import SentenceTransformer
    import numpy as np
    
    # 使用多语言模型（支持中文）
    # paraphrase-multilingual-MiniLM-L12-v2: 小而快，适合学习
    model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    
    # B站广告知识库文档
    knowledge_docs = [
        "广告 CTR 计算公式：点击数 / 展示数 × 100%",
        "CPM 是千次展示成本，计算方法：总消耗 / 展示数 × 1000",
        "B站开屏广告支持 5-15 秒视频或静态图片",
        "广告主可在广告投放平台实时查看消耗和效果数据",
        "OCPM 智能出价：系统自动优化出价，最大化转化量",
        "信息流广告支持按 CPM、CPC、OCPM 三种计费方式",
        "B站广告审核通常在 1 个工作日内完成",
    ]
    
    # 编码所有文档
    doc_embeddings = model.encode(knowledge_docs, normalize_embeddings=True)
    print(f"文档 Embedding 维度: {doc_embeddings.shape}")
    
    # 语义搜索
    queries = [
        "点击率怎么算",
        "广告费用计费模式",
        "审核需要多久",
    ]
    
    for query in queries:
        q_emb = model.encode([query], normalize_embeddings=True)[0]
        sims = doc_embeddings @ q_emb  # 余弦相似度（已 normalize）
        best_idx = sims.argmax()
        print(f"\n问题: {query!r}")
        print(f"最相关文档 (相似度={sims[best_idx]:.3f}): {knowledge_docs[best_idx]!r}")
        
except ImportError:
    print("sentence-transformers 未安装，展示逻辑示意")
    print("""
语义搜索流程（RAG 的核心）：

1. 离线阶段：
   docs = ["广告CTR公式...", "CPM计算...", ...]
   doc_embeddings = model.encode(docs)  # shape: [n_docs, 384]
   faiss_index.add(doc_embeddings)      # 存入向量库

2. 在线阶段：
   query = "点击率怎么算"
   q_emb = model.encode([query])        # shape: [1, 384]
   top_k = faiss_index.search(q_emb, k=3)  # 找最近的3个文档
   context = docs[top_k]
   answer = llm(query + context)        # LLM 结合 context 回答
    """)

## 4. Embedding 模型选型

| 模型 | 维度 | 最大长度 | 语言 | 场景 |
|------|------|---------|------|------|
| `text-embedding-3-small` (OpenAI) | 1536 | 8191 | 多语言 | 商业项目首选，效果好 |
| `text-embedding-3-large` (OpenAI) | 3072 | 8191 | 多语言 | 高精度场景 |
| `BAAI/bge-m3` | 1024 | 8192 | 多语言 | 本地部署最强，免费 |
| `BAAI/bge-large-zh` | 1024 | 512 | 中文 | 纯中文场景，本地 |
| `paraphrase-multilingual-MiniLM-L12-v2` | 384 | 128 | 多语言 | 学习/轻量场景 |
| `Qwen/Qwen3-Embedding` | 2048 | 32768 | 多语言 | 阿里云/本地，效果优秀 |

In [ ]:
# 演示如何调用 OpenAI Embedding API
import sys, os
sys.path.insert(0, "..")

def get_embedding_openai(text: str, model: str = "text-embedding-3-small") -> list[float]:
    """调用 OpenAI Embedding API"""
    import os
    from openai import OpenAI
    
    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
    response = client.embeddings.create(input=text, model=model)
    return response.data[0].embedding

def get_embedding_mock(text: str) -> np.ndarray:
    """Mock embedding（用于演示，无需 API key）"""
    # 用文本哈希模拟确定性 embedding
    rng = np.random.default_rng(hash(text) % (2**32))
    return rng.standard_normal(1536)

# 演示 Embedding 的使用方式
docs = [
    "广告CTR点击率计算",
    "CPM千次展示成本",
    "B站开屏广告格式",
]

# 检查是否有 API key
from dotenv import load_dotenv
load_dotenv("../.env")

if os.environ.get("OPENAI_API_KEY"):
    embeddings = [get_embedding_openai(d) for d in docs]
    print("使用 OpenAI text-embedding-3-small")
else:
    embeddings = [get_embedding_mock(d) for d in docs]
    print("使用 Mock embedding（演示模式，配置 OPENAI_API_KEY 后可用真实 embedding）")

emb_array = np.array(embeddings)
print(f"Embedding 矩阵 shape: {emb_array.shape}")

# 计算文档间相似度
norms = np.linalg.norm(emb_array, axis=1, keepdims=True)
normalized = emb_array / norms
sim_matrix = normalized @ normalized.T

print("\n文档相似度矩阵:")
for i, doc_i in enumerate(docs):
    for j, doc_j in enumerate(docs):
        if i < j:
            print(f"  sim({doc_i!r}, {doc_j!r}) = {sim_matrix[i,j]:.3f}")

## 5. 面试速记

| 问题 | 要点 |
|------|------|
| BPE 原理 | 字符出发，反复合并最频繁字符对，构建子词词表 |
| WordPiece vs BPE | WordPiece 用语言模型概率选择合并；BPE 用频率 |
| 为什么用 token 而不是字符 | 词汇表大小可控（5w vs 字符级几万），OOV 问题少 |
| Embedding 维度 | 常见 768/1024/1536/3072，越大表达力越强但计算越慢 |
| 语义搜索为什么用余弦相似度 | 对向量的长度不敏感，只关注方向（语义方向） |
| Embedding 和 Token Embedding 的区别 | Token Embedding 是 LLM 内部的（训练一起学）；Sentence Embedding 是独立编码器输出的整句向量 |

**下一节**: `03_llm_api_calling.ipynb` — OpenAI/Claude/通义千问 API 调用